# Food price pipeline: Bronze → Silver → Gold, quality flags, benchmark

Databricks notebook (serverless, Free Edition) for the Hyperlocal Food Price Intelligence Pipeline.
It uses `display()` and `dbutils`, so it runs on Databricks only.

**Prerequisite:** parquet files produced by `src/extract_mandi.py` are uploaded to the Volume
`workspace.food_price.raw_files` (created in the first cell). Run the cells in order.

| Layer | Tables |
|---|---|
| Bronze | `bronze_mandi` |
| Silver | `silver_mandi`, `silver_rejected`, `silver_mandi_v2` |
| Gold | `gold_market_daily`, `gold_peer_flags`, `gold_market_daily_flagged`, `gold_market_daily_clean` |
| Benchmark | `bench_labeled`, `bench_input`, `bench_result` |
| Demo | `silver_demo` (Delta MERGE) |
| Dashboard | `dash_*` |

## 0. Setup: schema and volume

In [1]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.food_price")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.food_price.raw_files")
print("OK")

OK


## 1. Bronze: raw parquet files to Delta
Raw data is kept exactly as received (all columns are strings), plus the source file path.

In [2]:
from pyspark.sql import functions as F

path = "/Volumes/workspace/food_price/raw_files/"

bronze = (
    spark.read.parquet(path + "*.parquet")
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.food_price.bronze_mandi"))

print("Bronze rows:", spark.table("workspace.food_price.bronze_mandi").count())

Bronze rows: 26676


In [3]:
display(
    spark.table("workspace.food_price.bronze_mandi")
    .groupBy("_source_file", "arrival_date")
    .count()
    .orderBy("_source_file")
)

_source_file,arrival_date,count
dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,30/10/2026,17115
dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-31_075532.parquet,31/10/2026,9561


Inspect the distinct state names (used to decide the state-name fixes in Silver).

In [4]:
b = spark.table("workspace.food_price.bronze_mandi")
display(b.select("state").distinct().orderBy("state"))

state
Andaman and Nicobar
Andhra Pradesh
Assam
Bihar
Chandigarh
Chattisgarh
Goa
Gujarat
Haryana
Himachal Pradesh


## 2. Silver v1: typed, deduplicated, validated
- dates parsed with `dd/MM/yyyy`; prices cast to double (bad values become null, not zero)
- exact duplicates removed (latest ingest wins)
- rows failing validation go to `silver_rejected` (quarantine) instead of being deleted
- prices converted from Rs/quintal to Rs/kg (1 quintal = 100 kg)

In [5]:
from pyspark.sql import functions as F, Window

SRC = "workspace.food_price.bronze_mandi"
b = spark.table(SRC)

# 1) Cast types and standardise names (raw columns are kept alongside)
p = (b
  .withColumn("arrival_dt", F.expr("try_to_timestamp(arrival_date, 'dd/MM/yyyy')").cast("date"))
  .withColumn("min_price_q",   F.expr("try_cast(min_price as double)"))
  .withColumn("max_price_q",   F.expr("try_cast(max_price as double)"))
  .withColumn("modal_price_q", F.expr("try_cast(modal_price as double)"))
  .withColumn("state_std",     F.initcap(F.trim("state")))
  .withColumn("district_std",  F.initcap(F.trim("district")))
  .withColumn("market_std",    F.trim("market"))
  .withColumn("commodity_std", F.initcap(F.trim("commodity")))
  .withColumn("variety_std",   F.initcap(F.trim("variety")))
  .withColumn("grade_std",     F.trim("grade"))
)

# state-name fixes found by inspecting the distinct state values
state_fix = {"Keralam": "Kerala", "Chattisgarh": "Chhattisgarh"}
p = p.replace(state_fix, subset=["state_std"])

# 2) Duplicates: same market+commodity+variety+grade+date; keep the latest ingest
key = ["state_std", "district_std", "market_std", "commodity_std",
       "variety_std", "grade_std", "arrival_dt"]
w = Window.partitionBy(*key).orderBy(F.col("_ingested_at").desc())
p = p.withColumn("_rn", F.row_number().over(w))

# 3) Validation: reject reason (missing prices are never treated as zero)
p = p.withColumn("reject_reason",
  F.when(F.col("arrival_dt").isNull(), "bad_date")
   .when(F.col("min_price_q").isNull() | F.col("max_price_q").isNull()
         | F.col("modal_price_q").isNull(), "missing_price")
   .when((F.col("min_price_q") <= 0) | (F.col("max_price_q") <= 0)
         | (F.col("modal_price_q") <= 0), "non_positive_price")
   .when(F.col("min_price_q") > F.col("max_price_q"), "min_gt_max")
   .when((F.col("modal_price_q") < F.col("min_price_q"))
         | (F.col("modal_price_q") > F.col("max_price_q")), "modal_out_of_range")
)

dedup = p.filter(F.col("_rn") == 1)
dups  = p.filter(F.col("_rn") > 1)

# 4) Unit: quintal -> kg (1 quintal = 100 kg)
valid = (dedup.filter(F.col("reject_reason").isNull())
  .withColumn("min_price_kg",   F.round(F.col("min_price_q")/100, 2))
  .withColumn("max_price_kg",   F.round(F.col("max_price_q")/100, 2))
  .withColumn("modal_price_kg", F.round(F.col("modal_price_q")/100, 2))
  .select("state_std","district_std","market_std","commodity_std","variety_std",
          "grade_std","arrival_dt","min_price_q","max_price_q","modal_price_q",
          "min_price_kg","max_price_kg","modal_price_kg",
          "_ingested_at","_source","_source_file")
)
rejected = dedup.filter(F.col("reject_reason").isNotNull())

valid.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
     .saveAsTable("workspace.food_price.silver_mandi")
rejected.drop("_rn").write.format("delta").mode("overwrite").option("overwriteSchema","true") \
     .saveAsTable("workspace.food_price.silver_rejected")

print("done")

done


### Measured metrics (real data)

In [6]:
raw   = spark.table(SRC).count()
slv   = spark.table("workspace.food_price.silver_mandi").count()
rej   = spark.table("workspace.food_price.silver_rejected")
rej_n = rej.count()
dup_n = raw - slv - rej_n

print("Raw rows:            ", raw)
print("Duplicates removed:  ", dup_n)
print("Rejected (invalid):  ", rej_n)
print("Silver (clean) rows: ", slv)

display(rej.groupBy("reject_reason").count())

s = spark.table("workspace.food_price.silver_mandi")
print("Dates:", [r[0] for r in s.select("arrival_dt").distinct().collect()])
print("States:", s.select("state_std").distinct().count(),
      "| Markets:", s.select("state_std","market_std").distinct().count(),
      "| Commodities:", s.select("commodity_std").distinct().count())
display(s.select("commodity_std","market_std","modal_price_q","modal_price_kg").limit(10))

Raw rows:             26676
Duplicates removed:   237
Rejected (invalid):   0
Silver (clean) rows:  26439


reject_reason,count


Dates: [datetime.date(2026, 10, 31), datetime.date(2026, 10, 30)]
States: 29 | Markets: 1900 | Commodities: 259


commodity_std,market_std,modal_price_q,modal_price_kg
Amaranthus,Diglipur Vegetable Market (Subhashgram),7000.0,70.0
Amaranthus,Diglipur Vegetable Market (Subhashgram),7000.0,70.0
Ashgourd,Diglipur Vegetable Market (Subhashgram),3500.0,35.0
Ashgourd,Diglipur Vegetable Market (Subhashgram),3500.0,35.0
Banana,Diglipur Vegetable Market (Subhashgram),3500.0,35.0
Banana,Diglipur Vegetable Market (Subhashgram),3500.0,35.0
Banana - Green,Diglipur Vegetable Market (Subhashgram),3500.0,35.0
Banana - Green,Diglipur Vegetable Market (Subhashgram),3500.0,35.0
Bhindi(ladies Finger),Diglipur Vegetable Market (Subhashgram),8000.0,80.0
Bhindi(ladies Finger),Diglipur Vegetable Market (Subhashgram),8000.0,80.0


## 3. Investigation: what are the duplicates, and are the prices sane?

In [7]:
from pyspark.sql import functions as F

# A) What kind of duplicates are these? (exact copies vs conflicting prices)
b = spark.table("workspace.food_price.bronze_mandi")
key = ["state","district","market","commodity","variety","grade","arrival_date"]
g = (b.groupBy(*key, "_source_file")
       .agg(F.count("*").alias("n"),
            F.countDistinct("min_price","max_price","modal_price").alias("distinct_prices")))
d = g.filter("n > 1")
print("Keys with duplicates:", d.count())
print("Exact duplicates (same price):", d.filter("distinct_prices = 1").count())
print("Conflicting (different price):", d.filter("distinct_prices > 1").count())
display(d.groupBy("_source_file").count())
display(d.orderBy(F.desc("n")).limit(10))

# B) Check the dates of the sample rows
s = spark.table("workspace.food_price.silver_mandi")
display(s.filter("market_std like 'Diglipur%' and commodity_std in ('Amaranthus','Ashgourd')")
         .select("commodity_std","arrival_dt","modal_price_q","modal_price_kg"))

# C) Price sanity: is any commodity's max far above its median?
display(s.groupBy("commodity_std").agg(
    F.count("*").alias("n"),
    F.expr("percentile_approx(modal_price_kg, 0.5)").alias("median_kg"),
    F.min("modal_price_kg").alias("min_kg"),
    F.max("modal_price_kg").alias("max_kg"))
  .withColumn("max_over_median", F.round(F.col("max_kg")/F.col("median_kg"), 1))
  .orderBy(F.desc("max_over_median")).limit(20))

Keys with duplicates: 237
Exact duplicates (same price): 237
Conflicting (different price): 0


_source_file,count
dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,237


state,district,market,commodity,variety,grade,arrival_date,_source_file,n,distinct_prices
Madhya Pradesh,Sagar,Bamora APMC,Lentil(Masur)(Whole),Local,FAQ,30/10/2026,dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,2,1
Madhya Pradesh,Shivpuri,Pohari APMC,Soyabean,Soyabeen,FAQ,30/10/2026,dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,2,1
Madhya Pradesh,Shivpuri,Pichhour APMC,Wheat,Mill Quality,Non-FAQ,30/10/2026,dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,2,1
Madhya Pradesh,Sagar,Deori APMC,Maize,Local,FAQ,30/10/2026,dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,2,1
Madhya Pradesh,Jabalpur,Jabalpur APMC,Paddy(Common),Dhan,FAQ,30/10/2026,dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,2,1
Madhya Pradesh,Shivpuri,Pohari APMC,Wheat,Wheat,Non-FAQ,30/10/2026,dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,2,1
Madhya Pradesh,Jabalpur,Jabalpur APMC,Maize,Local,FAQ,30/10/2026,dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,2,1
Madhya Pradesh,Shivpuri,Pohari APMC,"Sesamum(Sesame,Gingelly,Til)",Sesame,Non-FAQ,30/10/2026,dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,2,1
Madhya Pradesh,Shivpuri,Pohari APMC,Groundnut,Big (With Shell),Non-FAQ,30/10/2026,dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,2,1
Madhya Pradesh,Sagar,Bamora APMC,Maize,Local,FAQ,30/10/2026,dbfs:/Volumes/workspace/food_price/raw_files/mandi_2026-10-30_161121.parquet,2,1


commodity_std,arrival_dt,modal_price_q,modal_price_kg
Amaranthus,2026-10-30,7000.0,70.0
Amaranthus,2026-10-31,7000.0,70.0
Ashgourd,2026-10-30,3500.0,35.0
Ashgourd,2026-10-31,3500.0,35.0


commodity_std,n,median_kg,min_kg,max_kg,max_over_median
Cock,5,2.6,1.8,390.0,150.0
Wood,8,5.1,4.0,58.0,11.4
Green Chilli,688,48.0,0.0,450.0,9.4
Cashewnuts,9,130.0,100.0,1200.0,9.2
Marigold(loose),2,20.0,20.0,170.0,8.5
Tender Coconut,158,35.0,0.35,250.0,7.1
Coconut,369,57.5,0.3,270.0,4.7
Potato,751,18.0,0.0,82.5,4.6
Ginger(dry),14,65.0,55.0,280.0,4.3
Apple,244,85.0,0.3,355.0,4.2


## 4. Gold v1: market × commodity × day and a state-level peer comparison
First pass, kept because the investigation below builds on it. Superseded by the clean Gold in section 7.

In [8]:
s = spark.table("workspace.food_price.silver_mandi")

# Gold 1: market x commodity x day (average over varieties/grades)
daily = (s.groupBy("state_std","market_std","commodity_std","arrival_dt")
  .agg(F.round(F.avg("modal_price_kg"),2).alias("modal_kg"),
       F.count("*").alias("n_rows")))
daily.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
     .saveAsTable("workspace.food_price.gold_market_daily")

# Gold 2: peer comparison
k = ["state_std","commodity_std","arrival_dt"]
peer = daily.groupBy(*k).agg(F.count("*").alias("n_markets"),
          F.expr("percentile_approx(modal_kg,0.5)").alias("peer_median"))
dev = daily.join(peer, k).withColumn("abs_dev", F.abs(F.col("modal_kg")-F.col("peer_median")))
mad = dev.groupBy(*k).agg(F.expr("percentile_approx(abs_dev,0.5)").alias("mad"))

flags = (dev.join(mad, k)
  .withColumn("ratio_to_peer", F.round(F.col("modal_kg")/F.col("peer_median"),2))
  .withColumn("robust_z", F.when(F.col("mad")>0,
        F.round(0.6745*(F.col("modal_kg")-F.col("peer_median"))/F.col("mad"),2)))
  .withColumn("peer_outlier",
        (F.col("n_markets")>=5) & ((F.col("ratio_to_peer")>=2) | (F.col("ratio_to_peer")<=0.5))))

flags.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
     .saveAsTable("workspace.food_price.gold_peer_flags")

print("Market-commodity-day rows:", flags.count())
print("Peer outliers:", flags.filter("peer_outlier").count())
display(flags.filter("peer_outlier").orderBy(F.desc("ratio_to_peer")).limit(20))

Market-commodity-day rows: 25599
Peer outliers: 761


state_std,commodity_std,arrival_dt,market_std,modal_kg,n_rows,n_markets,peer_median,abs_dev,mad,ratio_to_peer,robust_z,peer_outlier
Maharashtra,Spinach,2026-10-31,APMC Kalmeshwar,13.15,1,6,0.13,13.02,0.06,101.15,146.37,true
Maharashtra,Spinach,2026-10-31,APMC Bhusaval,12.0,1,6,0.13,11.87,0.06,92.31,133.44,true
Maharashtra,Spinach,2026-10-31,Khed(Chakan),11.0,1,6,0.13,10.87,0.06,84.62,122.2,true
Maharashtra,Onion Green,2026-10-30,APMC Mumbai,10.5,1,5,0.3,10.2,0.21,35.0,32.76,true
Maharashtra,Onion Green,2026-10-30,APMC Solapur,6.0,1,5,0.3,5.7,0.21,20.0,18.31,true
Telangana,Cabbage,2026-10-31,Adilabad (Rythu Bazar),70.0,1,13,11.0,59.0,1.0,6.36,39.8,true
Telangana,Cabbage,2026-10-30,Adilabad (Rythu Bazar),70.0,1,15,11.0,59.0,4.25,6.36,9.36,true
Tamil Nadu,Lime,2026-10-31,Kariyapatti(Uzhavar Sandhai ),195.0,1,54,40.0,155.0,10.0,4.88,10.45,true
Tamil Nadu,Lime,2026-10-30,Aruppukottai(Uzhavar Sandhai ),195.0,1,56,40.0,155.0,10.0,4.88,10.45,true
Tamil Nadu,Lime,2026-10-31,Aruppukottai(Uzhavar Sandhai ),195.0,1,54,40.0,155.0,10.0,4.88,10.45,true


## 5. Quality flags v1 (national peers) and stale prices

In [9]:
from pyspark.sql import functions as F

s     = spark.table("workspace.food_price.silver_mandi")
daily = spark.table("workspace.food_price.gold_market_daily")

# National peers (more markets than state level, so the median is more stable)
nat = (daily.groupBy("commodity_std", "arrival_dt")
       .agg(F.count("*").alias("n_mkts_nat"),
            F.expr("percentile_approx(modal_kg, 0.5)").alias("nat_median")))

x = (daily.join(nat, ["commodity_std", "arrival_dt"])
     .withColumn("ratio_nat", F.round(F.col("modal_kg") / F.col("nat_median"), 3))
     .withColumn("quality_flag",
        F.when(F.col("modal_kg") < 0.5, "implausible_low")          # < Rs 0.5/kg
         .when(F.col("n_mkts_nat") < 10, "insufficient_peers")
         .when(F.col("ratio_nat").between(70, 140), "unit_x100_suspect")
         .when(F.col("ratio_nat").between(0.007, 0.014), "unit_div100_suspect")
         .when((F.col("ratio_nat") >= 3) | (F.col("ratio_nat") <= 0.33), "price_outlier")
         .otherwise("ok")))

x.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
 .saveAsTable("workspace.food_price.gold_market_daily_flagged")

display(x.groupBy("quality_flag").count().orderBy(F.desc("count")))
display(x.filter("quality_flag in ('implausible_low','unit_x100_suspect','unit_div100_suspect')")
         .groupBy("commodity_std", "quality_flag").count()
         .orderBy(F.desc("count")).limit(20))

# Stale price: same modal price on both days
st = (s.groupBy("state_std", "market_std", "commodity_std", "variety_std", "grade_std")
        .agg(F.countDistinct("arrival_dt").alias("days"),
             F.countDistinct("modal_price_q").alias("n_prices"))
        .filter("days = 2"))
tot, same = st.count(), st.filter("n_prices = 1").count()
print("Series seen on both days:", tot, "| unchanged price:", same, f"({same/tot:.0%})")

quality_flag,count
ok,24203
price_outlier,720
insufficient_peers,600
implausible_low,75
unit_div100_suspect,1


commodity_std,quality_flag,count
Spinach,implausible_low,8
Methi(leaves),implausible_low,6
Raddish,implausible_low,5
Banana,implausible_low,4
Lemon,implausible_low,4
Onion Green,implausible_low,4
Coriander(leaves),implausible_low,3
Coconut,implausible_low,3
Brinjal,implausible_low,3
Green Chilli,implausible_low,3


Series seen on both days: 8391 | unchanged price: 4905 (58%)


## 6. Investigation: implausibly low prices and unit patterns
Looks at which markets and items produce very low Rs/kg values, and whether retail-type markets behave differently.
Findings that fed the Silver v2 rules: the low values cluster in a few markets, `Wood` is timber and `Cock` is poultry (out of scope),
and some values are placeholders (for example 0.01).

In [10]:
from pyspark.sql import functions as F

fl  = spark.table("workspace.food_price.gold_market_daily_flagged")
s   = spark.table("workspace.food_price.silver_mandi")
low = fl.filter("quality_flag = 'implausible_low'")

# A) Raw quintal values of the low-price rows: kg-style numbers (5-50) or something else?
keys = ["state_std", "market_std", "commodity_std", "arrival_dt"]
raw = (s.join(low.select(*keys), keys)
        .select("state_std", "market_std", "commodity_std", "variety_std",
                "arrival_dt", "min_price_q", "max_price_q", "modal_price_q"))
display(raw.orderBy("commodity_std", "market_std").limit(50))

# B) Ratio to the national median
display(low.select("commodity_std", "market_std", "arrival_dt", "modal_kg",
                   "nat_median", "ratio_nat").orderBy("ratio_nat").limit(40))

# C) Where are they concentrated?
display(low.groupBy("state_std").count().orderBy(F.desc("count")))
display(low.groupBy("market_std").count().orderBy(F.desc("count")).limit(15))

# D) Do retail-type markets (Uzhavar/Rythu) have a different price level?
retail = fl.withColumn("retail_like",
    F.col("market_std").rlike("(?i)uzhavar|rythu|sandhai|bazar|bazaar"))
display(retail.groupBy("retail_like", "quality_flag").count()
        .orderBy("retail_like", F.desc("count")))

# E) Cock / Wood: which markets and rows?
display(s.filter("commodity_std in ('Cock', 'Wood')")
         .select("state_std", "market_std", "commodity_std", "variety_std",
                 "grade_std", "arrival_dt", "modal_price_q", "modal_price_kg")
         .orderBy("commodity_std", "modal_price_q"))

# F) Very small raw prices (< Rs 50 / quintal): how many and where?
tiny = s.filter("modal_price_q < 50")
print("Rows with modal_price_q < 50:", tiny.count())
display(tiny.groupBy("commodity_std").count().orderBy(F.desc("count")).limit(20))
display(tiny.select("state_std", "market_std", "commodity_std",
                    "min_price_q", "max_price_q", "modal_price_q")
            .orderBy("modal_price_q").limit(30))

state_std,market_std,commodity_std,variety_std,arrival_dt,min_price_q,max_price_q,modal_price_q
Kerala,Omalloor Market,Amaranthus,Amaranthus,2026-10-30,25.0,35.0,30.0
Madhya Pradesh,Jeerapur (F&V) APMC,Apple,Apple,2026-10-30,30.0,30.0,30.0
Kerala,Omalloor Market,Ashgourd,Ashgourd,2026-10-30,25.0,32.0,29.0
Madhya Pradesh,Jeerapur (F&V) APMC,Banana,Other,2026-10-30,15.0,15.0,15.0
Kerala,Omalloor Market,Banana,Palayamthodan,2026-10-30,7.0,10.0,8.0
Punjab,Patti APMC,Banana,Other,2026-10-30,0.04,0.04,0.04
Punjab,Patti APMC,Banana,Other,2026-10-31,0.05,0.05,0.05
Madhya Pradesh,Jeerapur (F&V) APMC,Bhindi(ladies Finger),Bhindi,2026-10-30,7.0,7.0,7.0
Madhya Pradesh,Jeerapur (F&V) APMC,Brinjal,Brinjal,2026-10-30,6.0,6.0,6.0
Punjab,Patti APMC,Brinjal,Other,2026-10-30,0.01,0.01,0.01


commodity_std,market_std,arrival_dt,modal_kg,nat_median,ratio_nat
Potato,Patti APMC,2026-10-30,0.0,15.0,0.0
Lemon,Diglipur Vegetable Market (Subhashgram),2026-10-31,0.03,170.0,0.0
Green Chilli,Patti APMC,2026-10-31,0.0,50.0,0.0
Brinjal,Patti APMC,2026-10-31,0.0,39.0,0.0
Cucumbar(kheera),Patti APMC,2026-10-30,0.0,30.0,0.0
Brinjal,Patti APMC,2026-10-30,0.0,32.5,0.0
Banana,Patti APMC,2026-10-31,0.0,50.0,0.0
Raddish,Patti APMC,2026-10-30,0.01,32.5,0.0
Green Chilli,Patti APMC,2026-10-30,0.0,47.5,0.0
Lemon,Pamohi(Garchuk) APMC,2026-10-31,0.06,170.0,0.0


state_std,count
Maharashtra,28
Punjab,18
Madhya Pradesh,9
Andaman And Nicobar,6
Goa,3
Telangana,3
Kerala,3
Karnataka,3
Assam,2


market_std,count
Patti APMC,16
Pune(Moshi),9
Jeerapur (F&V) APMC,9
Diglipur Vegetable Market (Subhashgram),6
Pune(Pimpri),5
APMC Akluj,4
Omalloor Market,3
APMC Pune,3
RYTHU BAZAR ERRAGADDA,3
Mapusa APMC,3


retail_like,quality_flag,count
false,ok,11620
false,price_outlier,617
false,insufficient_peers,517
false,implausible_low,72
false,unit_div100_suspect,1
true,ok,12583
true,price_outlier,103
true,insufficient_peers,83
true,implausible_low,3


state_std,market_std,commodity_std,variety_std,grade_std,arrival_dt,modal_price_q,modal_price_kg
Tripura,Masli,Cock,Boiler/firm(white),Local,2026-10-30,180.0,1.8
Tripura,Sonamura APMC,Cock,Boiler/firm(white),FAQ,2026-10-30,260.0,2.6
Tripura,Boxonagar,Cock,Boiler/firm(white),FAQ,2026-10-30,260.0,2.6
Nct Of Delhi,"Fish,Poultry & Egg Market, Gazipur APMC",Cock,Boiler/firm(white),Local,2026-10-30,7300.0,73.0
Nct Of Delhi,"Fish,Poultry & Egg Market, Gazipur APMC",Cock,Deshi,Local,2026-10-30,39000.0,390.0
Telangana,Dammapet APMC,Wood,Eucalyptus,FAQ,2026-10-31,400.0,4.0
Uttar Pradesh,Bhehjoi APMC,Wood,Eucalyptus,Local,2026-10-31,505.0,5.05
Uttar Pradesh,Bhehjoi APMC,Wood,Eucalyptus,Local,2026-10-30,505.0,5.05
Andhra Pradesh,Nandigama APMC,Wood,Subabul,Grade Range-1,2026-10-30,510.0,5.1
Andhra Pradesh,Jaggayyapeta APMC,Wood,Subabul,Grade Range-1,2026-10-30,550.0,5.5


Rows with modal_price_q < 50: 76


commodity_std,count
Spinach,8
Methi(leaves),6
Raddish,5
Banana,4
Onion Green,4
Lemon,4
Brinjal,4
Coriander(leaves),3
Mint(pudina),3
Coconut,3


state_std,market_std,commodity_std,min_price_q,max_price_q,modal_price_q
Punjab,Patti APMC,Onion,0.01,0.01,0.01
Punjab,Patti APMC,Brinjal,0.01,0.01,0.01
Punjab,Patti APMC,Onion,0.01,0.01,0.01
Punjab,Patti APMC,Brinjal,0.01,0.01,0.01
Punjab,Patti APMC,Potato,0.01,0.01,0.01
Punjab,Patti APMC,Green Chilli,0.01,0.01,0.01
Punjab,Patti APMC,Cucumbar(kheera),0.01,0.01,0.01
Punjab,Patti APMC,Brinjal,0.01,0.01,0.01
Punjab,Patti APMC,Tomato,0.01,0.01,0.01
Punjab,Patti APMC,Green Chilli,0.01,0.01,0.01


## 7. Silver v2 and clean Gold: plausibility flags
Rows are flagged, never deleted, and unit problems are evaluated at market level (median ratio of a market's items to the national median).
Unit-suspect rows are **not** auto-corrected because the true unit cannot be proven from the data.

In [11]:
from pyspark.sql import functions as F

s = spark.table("workspace.food_price.silver_mandi")

EXCLUDE   = ["Wood", "Cock"]   # Wood = timber, Cock = poultry
MIN_ITEMS = 3                  # minimum items needed for a market-level verdict

base = (s
  .withColumn("in_scope", ~F.col("commodity_std").isin(EXCLUDE))
  .withColumn("is_placeholder", F.col("modal_price_q") < 1))   # < Rs 1/quintal is not a real price

# 1) National median (quintal) per commodity-day
ref = base.filter("in_scope and not is_placeholder")
nat = (ref.groupBy("commodity_std", "arrival_dt")
          .agg(F.count("*").alias("n_nat"),
               F.expr("percentile_approx(modal_price_q, 0.5)").alias("nat_median_q")))

j = (base.join(nat, ["commodity_std", "arrival_dt"], "left")
         .withColumn("ratio_nat", F.col("modal_price_q") / F.col("nat_median_q")))

# 2) Market-level unit check: median ratio of all the market's items
evid = j.filter("in_scope and not is_placeholder and n_nat >= 10")
mk = (evid.groupBy("state_std", "market_std", "arrival_dt")
          .agg(F.count("*").alias("n_items"),
               F.expr("percentile_approx(ratio_nat, 0.5)").alias("market_ratio"))
          .filter(F.col("n_items") >= MIN_ITEMS)
          .withColumn("market_unit_suspect",
                      (F.col("market_ratio") < 0.05) | (F.col("market_ratio") > 20)))

# 3) Final flag (priority order)
v2 = (j.join(mk, ["state_std", "market_std", "arrival_dt"], "left")
  .withColumn("plausibility_flag",
     F.when(~F.col("in_scope"), "out_of_scope")
      .when(F.col("is_placeholder"), "placeholder_price")
      .when(F.col("market_unit_suspect"), "unit_suspect_market")
      .when((F.col("ratio_nat") < 0.02) | (F.col("ratio_nat") > 50), "unit_suspect_row")
      .otherwise("ok")))

v2.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
  .saveAsTable("workspace.food_price.silver_mandi_v2")

v = spark.table("workspace.food_price.silver_mandi_v2")
print("Silver v2 rows:", v.count())
display(v.groupBy("plausibility_flag").count().orderBy(F.desc("count")))

# Which markets were flagged
flagged = mk.filter("market_unit_suspect")
print("Market-days flagged unit_suspect:", flagged.count())
display(flagged.orderBy("market_ratio")
        .select("state_std", "market_std", "arrival_dt", "n_items", "market_ratio"))

# To sanity-check the threshold: distribution of market_ratio (10^bucket)
display(mk.withColumn("log10_bucket", F.floor(F.log10("market_ratio")))
          .groupBy("log10_bucket").count().orderBy("log10_bucket"))

# 4) Clean Gold: market-commodity-day from 'ok' rows only
ok = v.filter("plausibility_flag = 'ok'")
daily2 = (ok.groupBy("state_std", "market_std", "commodity_std", "arrival_dt")
            .agg(F.round(F.avg("modal_price_kg"), 2).alias("modal_kg"),
                 F.count("*").alias("n_rows")))
nat2 = (daily2.groupBy("commodity_std", "arrival_dt")
              .agg(F.count("*").alias("n_mkts_nat"),
                   F.expr("percentile_approx(modal_kg, 0.5)").alias("nat_median")))
g2 = (daily2.join(nat2, ["commodity_std", "arrival_dt"])
  .withColumn("ratio_nat", F.round(F.col("modal_kg") / F.col("nat_median"), 3))
  .withColumn("quality_flag",
     F.when(F.col("n_mkts_nat") < 10, "insufficient_peers")
      .when((F.col("ratio_nat") >= 3) | (F.col("ratio_nat") <= 0.33), "price_outlier")
      .otherwise("ok")))

g2.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
  .saveAsTable("workspace.food_price.gold_market_daily_clean")

print("Clean market-commodity-day rows:", g2.count())
display(g2.groupBy("quality_flag").count().orderBy(F.desc("count")))

Silver v2 rows: 26439


plausibility_flag,count
ok,26336
unit_suspect_row,47
unit_suspect_market,31
out_of_scope,13
placeholder_price,12


Market-days flagged unit_suspect: 4


state_std,market_std,arrival_dt,n_items,market_ratio
Madhya Pradesh,Jeerapur (F&V) APMC,2026-10-30,10,0.0036855036855036856
Kerala,Omalloor Market,2026-10-30,5,0.0145
Punjab,Patti APMC,2026-10-30,10,0.019656019656019656
Punjab,Patti APMC,2026-10-31,5,0.048


log10_bucket,count
-3,1
-2,4
-1,805
0,707


Clean market-commodity-day rows: 25498


quality_flag,count
ok,24199
price_outlier,712
insufficient_peers,587


## 8. Benchmark: inject labelled bugs
Bugs are injected into rows the rules consider clean (300 per type, seed 42). This measures the rules against known bugs;
it is **not** accuracy on real data.

In [12]:
from pyspark.sql import functions as F, Window

SEED, N = 42, 300
mkey = ["state_std", "market_std", "commodity_std", "arrival_dt"]
ukey = ["state_std", "district_std", "market_std", "commodity_std",
        "variety_std", "grade_std", "arrival_dt"]

v2  = spark.table("workspace.food_price.silver_mandi_v2")
gcl = spark.table("workspace.food_price.gold_market_daily_clean")

# Clean base: rows that the current rules consider 'ok'
bad = gcl.filter("quality_flag != 'ok'").select(*mkey).withColumn("_bad", F.lit(1))
base = (v2.filter("plausibility_flag = 'ok'")
          .join(bad, mkey, "left").filter("_bad is null")
          .select(*ukey, "min_price_q", "max_price_q", "modal_price_q"))
print("Clean base rows:", base.count())

b = base.withColumn("_rk", F.row_number().over(Window.orderBy(F.rand(SEED))))
types = ["dup_exact", "unit_x100", "unit_div100", "missing_price", "spike_small", "spike_big"]
e = None
for i, t in enumerate(types):
    cond = (F.col("_rk") > i * N) & (F.col("_rk") <= (i + 1) * N)
    e = F.when(cond, t) if e is None else e.when(cond, t)
b = b.withColumn("inj_type", e)

b = b.withColumn("f",
      F.when(F.col("inj_type") == "unit_x100", F.lit(100.0))
       .when(F.col("inj_type") == "unit_div100", F.lit(0.01))
       .when(F.col("inj_type") == "spike_small", 1.5 + F.rand(SEED + 1))    # 1.5x-2.5x
       .when(F.col("inj_type") == "spike_big", 3.0 + 5.0 * F.rand(SEED + 2))  # 3x-8x
       .otherwise(F.lit(1.0)))
for c in ["min_price_q", "max_price_q", "modal_price_q"]:
    b = b.withColumn(c, F.round(F.col(c) * F.col("f"), 2))
b = b.withColumn("modal_price_q",
      F.when(F.col("inj_type") == "missing_price", F.lit(None).cast("double"))
       .otherwise(F.col("modal_price_q")))

# materialise so the random injection is not re-evaluated
b.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
 .saveAsTable("workspace.food_price.bench_labeled")
b = spark.table("workspace.food_price.bench_labeled")

# duplicates: the original stays clean; its exact copy is added with the 'dup_exact' label
copies = b.filter("inj_type = 'dup_exact'")
orig = b.withColumn("inj_type",
         F.when(F.col("inj_type") == "dup_exact", F.lit(None).cast("string"))
          .otherwise(F.col("inj_type")))
test = (orig.unionByName(copies).drop("_rk")
            .withColumn("_rid", F.monotonically_increasing_id()))
test.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.food_price.bench_input")

display(spark.table("workspace.food_price.bench_input")
        .groupBy("inj_type").count().orderBy("inj_type"))

Clean base rows: 24982


inj_type,count
null,23482
dup_exact,300
missing_price,300
spike_big,300
spike_small,300
unit_div100,300
unit_x100,300


Run the same rules on the injected data and measure detection against the labels.

In [13]:
t = spark.table("workspace.food_price.bench_input")
EXCLUDE, MIN_ITEMS = ["Wood", "Cock"], 3

# dedup + validation
p = t.withColumn("_rn", F.row_number().over(Window.partitionBy(*ukey).orderBy("_rid")))
p = p.withColumn("reject_reason",
  F.when(F.col("min_price_q").isNull() | F.col("max_price_q").isNull()
         | F.col("modal_price_q").isNull(), "missing_price")
   .when((F.col("min_price_q") <= 0) | (F.col("max_price_q") <= 0)
         | (F.col("modal_price_q") <= 0), "non_positive_price")
   .when(F.col("min_price_q") > F.col("max_price_q"), "min_gt_max")
   .when((F.col("modal_price_q") < F.col("min_price_q"))
         | (F.col("modal_price_q") > F.col("max_price_q")), "modal_out_of_range"))
ded = p.filter("_rn = 1")
val = ded.filter("reject_reason is null")

# plausibility flags (same logic as Silver v2)
b2 = (val.withColumn("in_scope", ~F.col("commodity_std").isin(EXCLUDE))
         .withColumn("is_placeholder", F.col("modal_price_q") < 1))
nat = (b2.filter("in_scope and not is_placeholder")
         .groupBy("commodity_std", "arrival_dt")
         .agg(F.count("*").alias("n_nat"),
              F.expr("percentile_approx(modal_price_q, 0.5)").alias("nat_median_q")))
j = (b2.join(nat, ["commodity_std", "arrival_dt"], "left")
       .withColumn("ratio_nat", F.col("modal_price_q") / F.col("nat_median_q")))
mk = (j.filter("in_scope and not is_placeholder and n_nat >= 10")
        .groupBy("state_std", "market_std", "arrival_dt")
        .agg(F.count("*").alias("n_items"),
             F.expr("percentile_approx(ratio_nat, 0.5)").alias("market_ratio"))
        .filter(F.col("n_items") >= MIN_ITEMS)
        .withColumn("market_unit_suspect",
                    (F.col("market_ratio") < 0.05) | (F.col("market_ratio") > 20)))
v = (j.join(mk, ["state_std", "market_std", "arrival_dt"], "left")
      .withColumn("plausibility_flag",
        F.when(~F.col("in_scope"), "out_of_scope")
         .when(F.col("is_placeholder"), "placeholder_price")
         .when(F.col("market_unit_suspect"), "unit_suspect_market")
         .when((F.col("ratio_nat") < 0.02) | (F.col("ratio_nat") > 50), "unit_suspect_row")
         .otherwise("ok")))

# clean gold + price_outlier flag
d2 = (v.filter("plausibility_flag = 'ok'").groupBy(*mkey)
        .agg(F.round(F.avg(F.col("modal_price_q") / 100), 2).alias("modal_kg")))
n2 = (d2.groupBy("commodity_std", "arrival_dt")
        .agg(F.count("*").alias("n_mkts_nat"),
             F.expr("percentile_approx(modal_kg, 0.5)").alias("nat_median")))
g = (d2.join(n2, ["commodity_std", "arrival_dt"])
       .withColumn("ratio_nat", F.col("modal_kg") / F.col("nat_median"))
       .withColumn("gold_flag",
          F.when(F.col("n_mkts_nat") < 10, "insufficient_peers")
           .when((F.col("ratio_nat") >= 3) | (F.col("ratio_nat") <= 0.33), "price_outlier")
           .otherwise("ok"))
       .select(*mkey, "gold_flag"))

# row-level result
r = (ded.select("_rid", "inj_type", "f", "reject_reason", *mkey)
        .join(v.select("_rid", "plausibility_flag"), "_rid", "left")
        .join(g, mkey, "left")
        .withColumn("detected_by",
          F.when(F.col("reject_reason").isNotNull(), "validation_reject")
           .when(F.col("plausibility_flag") != "ok", "unit_or_scope_flag")
           .when(F.col("gold_flag") == "price_outlier", "price_outlier")
           .otherwise("missed")))
r.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
 .saveAsTable("workspace.food_price.bench_result")
r = spark.table("workspace.food_price.bench_result")

# ---- Metrics ----
print("Injected duplicate copies:", t.filter("inj_type = 'dup_exact'").count(),
      "| rows removed by dedup:", p.filter("_rn > 1").count())

inj   = r.filter("inj_type is not null and inj_type != 'dup_exact'")
clean = r.filter("inj_type is null")
tp, ni = inj.filter("detected_by != 'missed'").count(), inj.count()
fp, nc = clean.filter("detected_by != 'missed'").count(), clean.count()
print(f"Injected anomalies: {ni} | caught: {tp} | recall: {tp/ni:.3f}")
print(f"Clean rows: {nc} | false alarms: {fp} ({fp/nc:.3%}) | precision: {tp/(tp+fp):.3f}")

display(inj.groupBy("inj_type").agg(
    F.count("*").alias("n"),
    F.sum(F.when(F.col("detected_by") != "missed", 1).otherwise(0)).alias("caught"))
  .withColumn("recall", F.round(F.col("caught") / F.col("n"), 3)))
display(inj.groupBy("inj_type", "detected_by").count().orderBy("inj_type", "detected_by"))

# spike magnitude vs detection (threshold tradeoff)
sp = (inj.filter("inj_type in ('spike_small', 'spike_big')")
         .withColumn("mult",
            F.when(F.col("f") < 2, "1.5-2x").when(F.col("f") < 3, "2-3x")
             .when(F.col("f") < 5, "3-5x").otherwise("5-8x")))
display(sp.groupBy("mult").agg(
    F.count("*").alias("n"),
    F.sum(F.when(F.col("detected_by") != "missed", 1).otherwise(0)).alias("caught"))
  .withColumn("recall", F.round(F.col("caught") / F.col("n"), 3)).orderBy("mult"))

# Borderline market-days (ratio between 0.05 and 0.1) that were not flagged
real_mk = spark.table("workspace.food_price.silver_mandi_v2") \
    .filter("n_items is not null and not market_unit_suspect and market_ratio < 0.1") \
    .select("state_std", "market_std", "arrival_dt", "n_items", "market_ratio").distinct()
display(real_mk)

Injected duplicate copies: 300 | rows removed by dedup: 300
Injected anomalies: 1500 | caught: 1179 | recall: 0.786
Clean rows: 23482 | false alarms: 103 (0.439%) | precision: 0.920


inj_type,n,caught,recall
spike_big,300,252,0.84
spike_small,300,28,0.093
missing_price,300,300,1.0
unit_x100,300,299,0.997
unit_div100,300,300,1.0


inj_type,detected_by,count
missing_price,validation_reject,300
spike_big,missed,48
spike_big,price_outlier,252
spike_small,missed,272
spike_small,price_outlier,27
spike_small,unit_or_scope_flag,1
unit_div100,price_outlier,7
unit_div100,unit_or_scope_flag,293
unit_x100,missed,1
unit_x100,price_outlier,20


mult,n,caught,recall
1.5-2x,144,10,0.069
2-3x,156,18,0.115
3-5x,119,84,0.706
5-8x,181,168,0.928


state_std,market_std,arrival_dt,n_items,market_ratio
Madhya Pradesh,Morena APMC,2026-10-31,3,0.05172413793103448


## 9. Late-arriving corrections: Delta MERGE demo
Runs on a copy (`silver_demo`), so Silver v2 is untouched. The 50 corrections are simulated.

In [14]:
spark.sql("""CREATE OR REPLACE TABLE workspace.food_price.silver_demo AS
             SELECT * FROM workspace.food_price.silver_mandi_v2""")

# 50 corrected prices (simulated late corrections)
corr = (spark.table("workspace.food_price.silver_demo")
        .filter("plausibility_flag = 'ok' and arrival_dt = '2026-10-30'")
        .orderBy(F.rand(7)).limit(50)
        .select(*ukey,
                F.round(F.col("modal_price_q") * 1.1, 2).alias("new_modal_q"),
                F.current_timestamp().cast("string").alias("new_ingested_at")))
corr.createOrReplaceTempView("corrections")

spark.sql("""
MERGE INTO workspace.food_price.silver_demo t
USING corrections c
ON  t.state_std = c.state_std AND t.district_std = c.district_std
AND t.market_std = c.market_std AND t.commodity_std = c.commodity_std
AND t.variety_std = c.variety_std AND t.grade_std = c.grade_std
AND t.arrival_dt = c.arrival_dt
WHEN MATCHED AND c.new_ingested_at > t._ingested_at THEN UPDATE SET
  modal_price_q  = c.new_modal_q,
  modal_price_kg = round(c.new_modal_q / 100, 2),
  _ingested_at   = c.new_ingested_at
""")

display(spark.sql("DESCRIBE HISTORY workspace.food_price.silver_demo")
        .select("version", "timestamp", "operation", "operationMetrics"))

old = (spark.sql("SELECT * FROM workspace.food_price.silver_demo VERSION AS OF 0")
       .select(*ukey, F.col("modal_price_q").alias("old_q")))
new = (spark.table("workspace.food_price.silver_demo")
       .select(*ukey, F.col("modal_price_q").alias("new_q")))
chg = old.join(new, ukey).filter("old_q != new_q")
print("Rows corrected by MERGE:", chg.count(),
      "| total rows unchanged in count:", spark.table("workspace.food_price.silver_demo").count())
display(chg.limit(10))

version,timestamp,operation,operationMetrics
1,2026-10-31T14:49:00.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 12523, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 50, executionTimeMs -> 6301, materializeSourceTimeMs -> 1068, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2540, numTargetRowsUpdated -> 50, numOutputRows -> 50, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 50, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2581)"
0,2026-10-31T14:48:51.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 26439, numOutputBytes -> 419696)"


Rows corrected by MERGE: 50 | total rows unchanged in count: 26439


state_std,district_std,market_std,commodity_std,variety_std,grade_std,arrival_dt,old_q,new_q
Andhra Pradesh,Srikakulam,Narasannapet APMC,Paddy(common),Paddy,FAQ,2026-10-30,2369.0,2605.9
Bihar,Jamui,Jamui APMC,Green Chilli,Green Chilly,Grade A,2026-10-30,5800.0,6380.0
Gujarat,Banaskanth,Dhanera APMC,Bajra(pearl Millet/cumbu),Hybrid,Local,2026-10-30,2850.0,3135.0
Gujarat,Banaskanth,Tharad APMC,Isabgul(psyllium),Other,Local,2026-10-30,12125.0,13337.5
Gujarat,Rajkot,Dhoraji APMC,Groundnut,Hybrid,Non-FAQ,2026-10-30,6880.0,7568.0
Gujarat,Sabarkantha,Himatnagar(Veg.Market Himatnagar) APMC,Guar,Gwar,Local,2026-10-30,2200.0,2420.0
Gujarat,Surat,Bardoli(Madhi) APMC,Paddy(common),Other,Local,2026-10-30,2250.0,2475.0
Haryana,Fatehabad,Fatehabad APMC,Cucumbar(kheera),Other,Local,2026-10-30,3000.0,3300.0
Haryana,Fatehabad,Fatehabad APMC,Kinnow,Other,Local,2026-10-30,3000.0,3300.0
Haryana,Jhajar,Bahadurgarh APMC,Brinjal,Other,Medium,2026-10-30,1250.0,1375.0


## 10. Benchmark false-alarm check
Which rule produced the false alarms, and do they share a market-commodity-day with an injected row?

In [15]:
from pyspark.sql import functions as F
mkey = ["state_std", "market_std", "commodity_std", "arrival_dt"]

r = spark.table("workspace.food_price.bench_result")
fa = r.filter("inj_type is null and detected_by != 'missed'")
display(fa.groupBy("detected_by").count())

inj_keys = (r.filter("inj_type is not null and inj_type != 'dup_exact'")
              .select(*mkey).distinct().withColumn("shares_with_injected", F.lit(True)))
display(fa.join(inj_keys, mkey, "left")
          .fillna({"shares_with_injected": False})
          .groupBy("detected_by", "shares_with_injected").count())

# Morena APMC: borderline market (ratio just above the 0.05 cutoff)
display(spark.table("workspace.food_price.silver_mandi_v2")
  .filter("market_std like 'Morena%' and arrival_dt = '2026-10-31'")
  .select("commodity_std", "variety_std", "modal_price_q", "nat_median_q",
          "ratio_nat", "plausibility_flag"))

detected_by,count
price_outlier,103


detected_by,shares_with_injected,count
price_outlier,false,90
price_outlier,true,13


commodity_std,variety_std,modal_price_q,nat_median_q,ratio_nat,plausibility_flag
Potato,Desi,150.0,2900.0,0.05172413793103448,ok
Potato,Desi,150.0,2900.0,0.05172413793103448,ok
Sweet Potato,Sweet Potato,1300.0,4750.0,0.2736842105263158,ok


## 11. Tables for the dashboard

In [16]:
from pyspark.sql import functions as F

raw  = spark.table("workspace.food_price.bronze_mandi")
sil  = spark.table("workspace.food_price.silver_mandi_v2")
gold = spark.table("workspace.food_price.gold_market_daily_clean")
r    = spark.table("workspace.food_price.bench_result")
W = lambda df, name: (df.write.format("delta").mode("overwrite")
                        .option("overwriteSchema", "true")
                        .saveAsTable(f"workspace.food_price.{name}"))

# 1) Daily coverage + quality
raw_day = (raw.withColumn("arrival_dt",
             F.expr("try_to_timestamp(arrival_date, 'dd/MM/yyyy')").cast("date"))
              .groupBy("arrival_dt").agg(F.count("*").alias("raw_rows")))
cov = (sil.groupBy("arrival_dt").agg(
         F.count("*").alias("silver_rows"),
         F.countDistinct("state_std").alias("states"),
         F.countDistinct("state_std", "market_std").alias("markets"),
         F.countDistinct("commodity_std").alias("commodities"),
         F.sum(F.when(F.col("plausibility_flag") != "ok", 1).otherwise(0)).alias("flagged_rows")))
W(raw_day.join(cov, "arrival_dt"), "dash_daily_coverage")

# 2) Flag summary
W(sil.groupBy("plausibility_flag").count(), "dash_flag_summary")

# 3) Market-wise coverage by state
W(sil.groupBy("state_std", "arrival_dt")
     .agg(F.countDistinct("market_std").alias("markets"),
          F.countDistinct("commodity_std").alias("commodities")),
  "dash_state_coverage")

# 4) Cross-market price dispersion flags (>= 3x or <= 0.33x national median)
W(gold.filter("quality_flag = 'price_outlier'")
      .select("state_std", "market_std", "commodity_std", "arrival_dt",
              "modal_kg", "nat_median", "ratio_nat", "n_mkts_nat"),
  "dash_price_flags")

# 5) Benchmark results
inj = r.filter("inj_type is not null and inj_type != 'dup_exact'")
det = F.sum(F.when(F.col("detected_by") != "missed", 1).otherwise(0))
W(inj.groupBy("inj_type").agg(F.count("*").alias("n"), det.alias("caught"))
     .withColumn("recall", F.round(F.col("caught") / F.col("n"), 3)),
  "dash_benchmark")
sp = (inj.filter("inj_type in ('spike_small','spike_big')")
         .withColumn("multiplier",
            F.when(F.col("f") < 2, "1.5-2x").when(F.col("f") < 3, "2-3x")
             .when(F.col("f") < 5, "3-5x").otherwise("5-8x")))
W(sp.groupBy("multiplier").agg(F.count("*").alias("n"), det.alias("caught"))
    .withColumn("recall", F.round(F.col("caught") / F.col("n"), 3)),
  "dash_spike_by_size")

# 6) National median for key items (clean rows)
key_items = ["Tomato", "Onion", "Potato", "Green Chilli", "Cauliflower", "Cabbage"]
W(gold.filter(F.col("commodity_std").isin(key_items))
      .groupBy("commodity_std", "arrival_dt")
      .agg(F.expr("percentile_approx(modal_kg, 0.5)").alias("median_kg"),
           F.count("*").alias("markets")),
  "dash_key_items")

print("dashboard tables ready")
display(spark.table("workspace.food_price.dash_daily_coverage"))

dashboard tables ready


arrival_dt,raw_rows,silver_rows,states,markets,commodities,flagged_rows
2026-10-31,9561,9561,24,804,172,32
2026-10-30,17115,16878,29,1804,257,71
